In [154]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

In [155]:
odr = pd.read_csv("data/age-dependency-ratio-old.csv")
tfp = pd.read_csv("data/total-factor-productivity.csv") 
schooling = pd.read_csv("data/average-years-of-schooling-among-adults.csv")
hc = pd.read_csv("data/HC.csv")

# Gør HC fra wide til long format
hc_long = hc.melt(
    id_vars=["ISO code", "Country", "Variable code", "Variable name"],
    var_name="year",
    value_name="hc"
)

# Omdøb kolonner så de matcher dine andre datasæt
hc_long = hc_long.rename(columns={
    "ISO code": "code",
    "Country": "country"
})

# Gør year numerisk
hc_long["year"] = pd.to_numeric(hc_long["year"], errors="coerce")

# Fjern rækker uden år eller HC-værdi
hc_long = hc_long.dropna(subset=["year", "hc"])

# Gør year til integer
hc_long["year"] = hc_long["year"].astype(int)

# Begræns perioden, hvis du vil
hc_long = hc_long[(hc_long["year"] >= 1990) & (hc_long["year"] <= 2020)]

# Behold kun de nødvendige kolonner
hc_long = hc_long[["code", "year", "hc"]]

In [156]:
odr = odr.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "odr"
})

tfp = tfp.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Total factor productivity level": "tfp"
})

schooling = schooling.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Both genders": "schooling"
})

hc_long.head(5)

,code,year,hc
5800,AGO,1990,1.138071
5801,ALB,1990,2.516159
5802,ARE,1990,2.014390
5803,ARG,1990,2.529030
5804,ARM,1990,2.948581


In [157]:

panel2 = tfp

panel2 = panel2.merge(
    schooling[["code", "year", "schooling"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    odr[["code", "year", "odr"]],
    on=["code", "year"],
    how="inner"
)

panel2 = panel2.merge(
    hc_long[["code", "year", "hc"]],
    on=["code", "year"],
    how="inner"
)

panel2["log_tfp"] = np.log(panel2["tfp"])



In [158]:
panel2["ODRxHC"] = panel2["odr"] * panel2["hc"]

In [159]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model3 = smf.ols(
    "log_tfp ~ odr + hc + ODRxHC + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "hc", "ODRxHC"]],
    "std_err": results3.bse[["odr", "hc", "ODRxHC"]],
    "p_value": results3.pvalues[["odr", "hc", "ODRxHC"]],
})

print(main_results3.round(4))

          coef  std_err  p_value
odr    -0.0179   0.0250   0.4757
hc     -0.2940   0.1369   0.0317
ODRxHC  0.0061   0.0066   0.3543


In [160]:
# estimation
model2 = smf.ols(
    "log_tfp ~ odr + hc + C(code) + C(year)",
    data=df_est
)

results2 = model2.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results2 = pd.DataFrame({
    "coef": results2.params[["odr", "hc"]],
    "std_err": results2.bse[["odr", "hc"]],
    "p_value": results2.pvalues[["odr", "hc"]],
})

print(main_results2.round(4))

       coef  std_err  p_value
odr  0.0043   0.0059   0.4582
hc  -0.2600   0.1300   0.0456


In [161]:
# estimation
model1 = smf.ols(
    "log_tfp ~ odr + C(code) + C(year)",
    data=df_est
)

results1 = model1.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results1 = pd.DataFrame({
    "coef": results1.params[["odr"]],
    "std_err": results1.bse[["odr"]],
    "p_value": results1.pvalues[["odr"]],
})

print(main_results1.round(4))

       coef  std_err  p_value
odr  0.0043   0.0059   0.4614


In [162]:
results = summary_col(
    [results1, results2, results3],
    stars=True,
    model_names=['(1)', '(2)', '(3)'],
    regressor_order=['odr', 'hc', 'ODRxHC'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                 (1)       (2)       (3)   
-------------------------------------------
odr            0.0043   0.0043    -0.0179  
               (0.0059) (0.0059)  (0.0250) 
hc                      -0.2600** -0.2940**
                        (0.1300)  (0.1369) 
ODRxHC                            0.0061   
                                  (0.0066) 
R-squared      0.7326   0.7393    0.7404   
R-squared Adj. 0.7209   0.7279    0.7289   
N              3545     3545      3545     
R2             0.733    0.739     0.740    
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


In [163]:
# kopi af det færdige balanced panel
df_est = panel2.copy()

# estimation
model6 = smf.ols(
    "tfp ~ odr + hc + ODRxHC + C(code) + C(year)",
    data=df_est
)

results6 = model6.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results6 = pd.DataFrame({
    "coef": results6.params[["odr", "hc", "ODRxHC"]],
    "std_err": results6.bse[["odr", "hc", "ODRxHC"]],
    "p_value": results6.pvalues[["odr", "hc", "ODRxHC"]],
})

print(main_results6.round(4))

          coef  std_err  p_value
odr    -0.0283   0.0236   0.2293
hc     -0.3710   0.1706   0.0297
ODRxHC  0.0099   0.0067   0.1386


In [164]:
# estimation
model5 = smf.ols(
    "tfp ~ odr + hc + C(code) + C(year)",
    data=df_est
)

results5 = model5.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results5 = pd.DataFrame({
    "coef": results5.params[["odr", "hc"]],
    "std_err": results5.bse[["odr", "hc"]],
    "p_value": results5.pvalues[["odr", "hc"]],
})

print(main_results5.round(4))

       coef  std_err  p_value
odr  0.0075   0.0059   0.2009
hc  -0.3160   0.1552   0.0417


In [165]:
# estimation
model4 = smf.ols(
    "tfp ~ odr + C(code) + C(year)",
    data=df_est
)

results4 = model4.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results4 = pd.DataFrame({
    "coef": results4.params[["odr"]],
    "std_err": results4.bse[["odr"]],
    "p_value": results4.pvalues[["odr"]],
})

print(main_results4.round(4))

       coef  std_err  p_value
odr  0.0075   0.0059   0.2032


In [166]:
results = summary_col(
    [results4, results5, results6],
    stars=True,
    model_names=['(1)', '(2)', '(3)'],
    regressor_order=['odr', 'hc', 'ODRxHC'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                 (1)       (2)       (3)   
-------------------------------------------
odr            0.0075   0.0075    -0.0283  
               (0.0059) (0.0059)  (0.0236) 
hc                      -0.3160** -0.3710**
                        (0.1552)  (0.1706) 
ODRxHC                            0.0099   
                                  (0.0067) 
R-squared      0.8107   0.8175    0.8194   
R-squared Adj. 0.8024   0.8095    0.8114   
N              3545     3545      3545     
R2             0.811    0.817     0.819    
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01
